### Point Critical Infrastructure Data to Hexbin

In [ ]:
import os

os.chdir("")

os.getcwd()

In [1]:
import numpy as np
from shapely.geometry import Point, Polygon
import pandas as pd
import matplotlib.pyplot as plt
import folium
import base64
import geopandas as gpd
import pathlib as Path
import geopandas as gpd
from shapely import union_all
import fiona
#import gdal


In [ ]:
# Starting by taking the H3 Master and clipping to the buffered PlanRVA Boundary
# You can skip this if you have H3 cells already covering the area you want to map correctly.
#To skip, simply read the H3 geopackage you want to start with and being loading the point data.

# Clip to whatever project boundary you have - 
# Reading the h3 master resolution 8 as a geopackage
# Reading the buffer as a polygon shapefile

hex_path = ".gpkg"
#Define GPD:
gdf_hex = gpd.read_file(hex_path)

#Read the Buffer Around the Region
planrva = gpd.read_file(r'.shp')

In [5]:
# Make planrva into same crs as hex
planrva = planrva.to_crs(gdf_hex.crs)

In [ ]:
# Check tthe crs again
print(gdf_hex.crs), print(planrva.crs)

In [ ]:
# Clip hex to buffer
hex_buff = gdf_hex[gdf_hex.intersects(planrva)]
# I want to save to check
hex_buff.to_file(r'.gpkg',
    driver="GPKG")

In [ ]:
#Read the infrastructure point data
#This is cleaned infrastructure point data - with Sector and Sub-Sector names
#These were all shapefiles
commercial = gpd.read_file(r'')
communication = gpd.read_file(r'')
dams = gpd.read_file(r'')
emergency = gpd.read_file(r'')
energy = gpd.read_file(r'')
financial = gpd.read_file(r'')
food = gpd.read_file(r'')
government = gpd.read_file(r'')
health = gpd.read_file(r'')
nuclear = gpd.read_file(r'')
transportation = gpd.read_file(r'')
water = gpd.read_file(r'')

In [ ]:
#clip to hex buff area
commercial_clip = gpd.clip(commercial_proj, hex_buff_proj)
communication_clip = gpd.clip(communication_proj, hex_buff_proj)
dams_clip = gpd.clip(dams_proj, hex_buff_proj)
emergency_clip = gpd.clip(emergency_proj, hex_buff_proj)
energy_clip = gpd.clip(energy_proj, hex_buff_proj)
financial_clip = gpd.clip(financial_proj, hex_buff_proj)
food_clip = gpd.clip(food_proj, hex_buff_proj)
government_clip = gpd.clip(government_proj, hex_buff_proj)
health_clip = gpd.clip(health_proj, hex_buff_proj)
nuclear_clip = gpd.clip(nuclear_proj, hex_buff_proj)
transportation_clip = gpd.clip(transportation_proj, hex_buff_proj)
water_clip = gpd.clip(water_proj, hex_buff_proj)

In [ ]:
#ensure common CRS
projected_crs = "EPSG:2284"

hex_buff_proj = hex_buff.to_crs(projected_crs)

commercial_proj = commercial.to_crs(projected_crs)
communication_proj = communication.to_crs(projected_crs)
dams_proj = dams.to_crs(projected_crs)
emergency_proj = emergency.to_crs(projected_crs)
energy_proj = energy.to_crs(projected_crs)
financial_proj = financial.to_crs(projected_crs)
food_proj = food.to_crs(projected_crs)
government_proj = government.to_crs(projected_crs)
health_proj = health.to_crs(projected_crs)
nuclear_proj = nuclear.to_crs(projected_crs)
transportation_proj = transportation.to_crs(projected_crs)
water_proj = water.to_crs(projected_crs)

energy_line_proj = energy_line.to_crs(projected_crs)
trans_line_proj = trans_line.to_crs(projected_crs)

In [ ]:
#create list of point data
gdfs_point = [commercial_clip, communication_clip, dams_clip, emergency_clip, financial_clip, 
              food_clip, government_clip, health_clip, nuclear_clip, transportation_clip, 
              water_clip]


# Combine into one GeoDataFrame
combined_points = gpd.GeoDataFrame(
    pd.concat(gdfs_point, ignore_index=True),
    crs=gdfs_point[0].crs
)

In [ ]:
#Read all the point Data 
folder = "\Point_Data_Cleaned"

#create empty list
gdfs = []

for file in os.listdir(folder):
    if file.endswith(".shp"):
        full_path = os.path.join(folder, file)
        gdf = gpd.read_file(full_path)
        gdfs.append(gdf)
# Ensure same crs
gdfs = [gdf.to_crs("EPSG:4326") for gdf in gdfs]
# Combine into one GeoDataFrame
combined_gdf = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True))
# Ensure same crs
combined_gdf = [combined_gdf.to_crs("EPSG:4326")]
# Ensure it is a geodataframe correctly - was a list before
combined_gdf = gpd.GeoDataFrame(
    pd.concat(gdfs, ignore_index=True),
    crs=gdfs[0].crs
)

In [ ]:
#Checking for geodataframe and not list
type(combined_gdf)

_______________________________________

In [ ]:
#Read the hex buff
hex_buff = gpd.read_file(r'.gpkg')
#reread combined gdf
combined_gdf = gpd.read_file(r'\combined_points.shp')

#ensure common CRS
projected_crs = "EPSG:2284"

combined_gdf = combined_gdf.to_crs(projected_crs)
hex_buff = hex_buff.to_crs(projected_crs)

#From the combined_gdf, want to keep counts of sector and sub-sector 
#Find out how they overlay
joined = gpd.sjoin(
    combined_gdf,
    hex_buff,
    how="inner",
    predicate="within"   # or "intersects"
)

#create counts
subsector_counts = (
    joined
    .groupby(["h3_ID", "Sub-Sector"])
    .size()
    .reset_index(name="count")
)

subsector_wide = subsector_counts.pivot(
    index="h3_ID",
    columns="Sub-Sector",
    values="count"
).fillna(0).reset_index()


# Merge sub-sector totals
hex_summary = hex_buff.merge(subsector_wide, on="h3_ID", how="left")

# Replace NaNs with 0
hex_summary = hex_summary.fillna(0)


#Save to file
#name and save to folder with H3 Edits
hex_summary.to_file(r'.gpkg',
    driver="GPKG")

In [ ]:
#lets check this
hex_summary['Local Emergency Shelters'].describe()

In [ ]:
#checking Groupby results
hex_summary.head()

In [ ]:
#Save to file
hex_summary.to_file(
    r'.gpkg',
    layer="hex_sum",
    driver="GPKG"
)

In [27]:
#let's visualize
import folium

In [ ]:
hex_summary.explore()

In [ ]:
#After review in ArcGIS, we need to read the geopackage again and foce columns to an integer type
gpkg_path = r".gpkg"

gdf = gpd.read_file(gpkg_path)

print(gdf.dtypes)

In [11]:
for col in gdf.select_dtypes(include=["float64"]).columns:
    if (gdf[col] % 1 == 0).all():
        gdf[col] = gdf[col].astype("Int64")

In [ ]:
#Let's check this
print(gdf.dtypes)

In [ ]:
#check various sub sectors
total = gdf['Dam or Levee'].sum()
print(total)

In [ ]:
#Save to file
#name and save to folder with H3 Edits - Overwrite 
gdf.to_file(r'.gpkg')

In [ ]:
gdf.to_excel(r'.xlsx')